# 01 — Preprocessing, Visualisation and Pretrained Baseline
**AI-Based Pancreatic Cancer Prediction Using Deep Learning (Early Detection from Medical Imaging Data)**
Phase-II · Review 1 · AITM Bhatkal, Dept. of CSE · Team: Hasan, Mustafa, Wasih · Guide: Mrs. Arzoo

This notebook implements and evidences:
- **Stage 6.1 — Data Preprocessing** (resampling, RAS orientation, HU windowing, normalisation, patient-wise split, augmentation)
- **Stage 6.2 — ROI Segmentation, baseline**: a pretrained pancreas + tumor segmentation model (MONAI `pancreas_ct_dints_segmentation`, trained on MSD Task07) run on held-out cases and scored against ground truth.

Dataset for this stage: **MSD Task07 Pancreas** (281 labelled portal-venous CTs, pancreas + tumor masks).
PanTS (36,390 CTs) is staged separately on the college machine for Review 2; it is too large for this session.

Runtime: **Colab, GPU (T4)**. `Runtime → Change runtime type → T4 GPU` before running.


In [ ]:
!nvidia-smi -L
import torch; print("CUDA:", torch.cuda.is_available())

## 0. Setup

In [ ]:
%%capture
!pip install -q "monai[nibabel,skimage,tqdm,pyyaml]" pytorch-ignite pandas matplotlib

In [ ]:
import os, json, glob, random, time
import numpy as np, pandas as pd, nibabel as nib
import matplotlib.pyplot as plt
import monai
from monai.transforms import (Compose, LoadImaged, EnsureChannelFirstd, Orientationd, Spacingd,
                              ScaleIntensityRanged, CropForegroundd, EnsureTyped,
                              RandFlipd, RandRotate90d, Rand3DElasticd)
monai.config.print_config()

# Optional: mount Google Drive so results survive a Colab disconnect
SAVE_TO_DRIVE = True
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RESULTS = '/content/drive/MyDrive/pancreas_project/results/review1'
else:
    RESULTS = '/content/results/review1'
os.makedirs(RESULTS, exist_ok=True)
print("Results ->", RESULTS)

## 1. Download MSD Task07 Pancreas (≈12 GB, into Colab's own disk)

In [ ]:
from monai.apps import download_and_extract
DATA_ROOT = '/content/data'
os.makedirs(DATA_ROOT, exist_ok=True)
TASK_DIR = f'{DATA_ROOT}/Task07_Pancreas'
if not os.path.exists(TASK_DIR):
    url = 'https://msd-for-monai.s3-us-west-2.amazonaws.com/Task07_Pancreas.tar'
    download_and_extract(url, f'{DATA_ROOT}/Task07_Pancreas.tar', DATA_ROOT)
print(os.listdir(TASK_DIR))
ds_meta = json.load(open(f'{TASK_DIR}/dataset.json'))
print(ds_meta['name'], '|', ds_meta['description'])
print('labels:', ds_meta['labels'])
print('training cases:', ds_meta['numTraining'], '| test cases:', ds_meta['numTest'])

## 2. Patient-wise subset and split (Stage 6.1)
Each MSD case is one patient, so a case-level split is a patient-wise split — no patient appears in both sets.

In [ ]:
random.seed(42)
all_cases = [{'image': f"{TASK_DIR}/{d['image'].lstrip('./')}",
              'label': f"{TASK_DIR}/{d['label'].lstrip('./')}"} for d in ds_meta['training']]
N_SUBSET = 20                      # small subset for tonight; full 281 on the college GPU
subset = random.sample(all_cases, N_SUBSET)
n_val = 4
train_files, val_files = subset[:-n_val], subset[-n_val:]
print(f'subset={len(subset)}  train={len(train_files)}  val={len(val_files)}')
json.dump({'train': train_files, 'val': val_files}, open(f'{RESULTS}/split_review1.json', 'w'), indent=1)
[os.path.basename(f['image']) for f in val_files]

## 3. Raw data statistics — tumor size and the sub-2 cm question
Tumor volume is computed from the label mask; the equivalent spherical diameter tells us how many cases fall in the **sub-2 cm** range the project targets.

In [ ]:
rows = []
for f in subset:
    img = nib.load(f['image']); lab = nib.load(f['label'])
    sp = img.header.get_zooms()[:3]
    L = np.asarray(lab.dataobj)
    vox_ml = np.prod(sp) / 1000.0
    panc = int((L >= 1).sum()); tum = int((L == 2).sum())
    tum_ml = tum * vox_ml
    eq_diam_cm = (6 * tum_ml / np.pi) ** (1/3) if tum > 0 else 0.0
    rows.append(dict(case=os.path.basename(f['image']).replace('.nii.gz',''),
                     shape=str(img.shape), spacing_mm=str(tuple(round(s,2) for s in sp)),
                     pancreas_vox=panc, tumor_vox=tum, tumor_ml=round(tum_ml,2),
                     eq_diam_cm=round(eq_diam_cm,2), sub2cm=eq_diam_cm < 2.0))
stats = pd.DataFrame(rows)
stats.to_csv(f'{RESULTS}/raw_stats_subset.csv', index=False)
print(f"Cases with tumor: {(stats.tumor_vox>0).sum()}/{len(stats)} | sub-2 cm: {stats.sub2cm.sum()}")
stats

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
stats['eq_diam_cm'].plot.hist(bins=10, ax=ax[0], color='#5b4fbf'); ax[0].axvline(2.0, color='r', ls='--')
ax[0].set_title('Tumor equivalent diameter (cm)'); ax[0].set_xlabel('cm')
stats[['pancreas_vox','tumor_vox']].plot.bar(ax=ax[1], logy=True); ax[1].set_title('Voxels per case (log)')
plt.tight_layout(); plt.savefig(f'{RESULTS}/fig_raw_stats.png', dpi=150); plt.show()

## 4. Preprocessing pipeline (Stage 6.1, as specified in the report)
| Report step | Implementation |
|---|---|
| Uniform voxel spacing | `Spacingd(1.5, 1.5, 2.0 mm)` |
| Consistent orientation | `Orientationd(RAS)` |
| HU clip to pancreatic soft-tissue window | `ScaleIntensityRanged(-100 … 240 HU → 0 … 1)` |
| Intensity normalisation | same transform (min-max to [0,1]) |
| Augmentation (train only) | random flip, 90° rotation, elastic deformation |

In [ ]:
SPACING = (1.5, 1.5, 2.0)
HU_MIN, HU_MAX = -100, 240

base = [
    LoadImaged(keys=['image','label']),
    EnsureChannelFirstd(keys=['image','label']),
    Orientationd(keys=['image','label'], axcodes='RAS'),
    Spacingd(keys=['image','label'], pixdim=SPACING, mode=('bilinear','nearest')),
    ScaleIntensityRanged(keys=['image'], a_min=HU_MIN, a_max=HU_MAX, b_min=0.0, b_max=1.0, clip=True),
    CropForegroundd(keys=['image','label'], source_key='image'),
    EnsureTyped(keys=['image','label']),
]
augment = [
    RandFlipd(keys=['image','label'], prob=0.5, spatial_axis=0),
    RandRotate90d(keys=['image','label'], prob=0.5, max_k=3, spatial_axes=(0,1)),
    Rand3DElasticd(keys=['image','label'], prob=1.0, sigma_range=(5,8), magnitude_range=(50,120),
                   mode=('bilinear','nearest'), padding_mode='zeros'),
]
val_tf   = Compose(base)
train_tf = Compose(base + augment)
print('val transforms  :', len(val_tf.transforms))
print('train transforms:', len(train_tf.transforms), '(incl. augmentation)')

In [ ]:
def best_axial(label):  # slice index with the largest tumor area, else largest pancreas area
    L = np.asarray(label)[0]
    area = (L == 2).sum(axis=(0,1))
    if area.max() == 0: area = (L >= 1).sum(axis=(0,1))
    return int(area.argmax())

def show_case(d, title, fname=None):
    img = np.asarray(d['image'])[0]; lab = np.asarray(d['label'])[0]
    z = best_axial(d['label'])
    im, lb = np.rot90(img[:,:,z]), np.rot90(lab[:,:,z])
    fig, ax = plt.subplots(1, 3, figsize=(12, 4))
    ax[0].imshow(im, cmap='gray'); ax[0].set_title('preprocessed CT (windowed, resampled)')
    ax[1].imshow(im, cmap='gray'); ax[1].imshow(np.ma.masked_where(lb==0, lb), cmap='spring', alpha=0.5); ax[1].set_title('pancreas (1) + tumor (2) overlay')
    ax[2].imshow(im, cmap='gray'); ax[2].contour(lb>=1, colors='lime', linewidths=0.8); ax[2].contour(lb==2, colors='red', linewidths=0.8); ax[2].set_title('contours: green=pancreas, red=tumor')
    for a in ax: a.axis('off')
    fig.suptitle(f'{title} | slice z={z} | shape {img.shape} | spacing {SPACING} mm', fontsize=10)
    plt.tight_layout()
    if fname: plt.savefig(fname, dpi=150)
    plt.show()

for f in val_files:
    d = val_tf(dict(f))
    name = os.path.basename(f['image']).replace('.nii.gz','')
    show_case(d, name, f'{RESULTS}/overlay_{name}.png')

### Augmentation check (train transforms only)

In [ ]:
f = train_files[0]; name = os.path.basename(f['image']).replace('.nii.gz','')
show_case(val_tf(dict(f)), f'{name} — original')
show_case(train_tf(dict(f)), f'{name} — augmented (flip / rot90 / elastic)', f'{RESULTS}/augment_{name}.png')

## 5. Pretrained baseline for ROI segmentation (Stage 6.2)
MONAI model-zoo bundle **`pancreas_ct_dints_segmentation`** — a 3-class (background / pancreas / tumor) 3D model trained on MSD Task07, reported mean Dice ≈ 0.62. We run it on our 4 held-out validation cases and score it.
This establishes the "existing system" baseline the Attention U-Net (Review 2) must beat.

In [ ]:
BUNDLE_DIR = '/content/bundles'
!python -m monai.bundle download --name pancreas_ct_dints_segmentation --bundle_dir {BUNDLE_DIR}
BUNDLE = f'{BUNDLE_DIR}/pancreas_ct_dints_segmentation'
print(os.listdir(BUNDLE)); print(os.listdir(f'{BUNDLE}/models'))

In [ ]:
# datalist in the format the bundle expects ('testing' key, paths relative to dataset_dir)
datalist = {'testing': [{'image': os.path.relpath(f['image'], TASK_DIR)} for f in val_files]}
DL_PATH = f'{BUNDLE}/configs/datalist_review1.json'
json.dump(datalist, open(DL_PATH, 'w'), indent=1)
PRED_DIR = '/content/baseline_pred'
print(json.dumps(datalist, indent=1))

In [ ]:
%%time
# If this OOMs on a T4, add:  --inferer#sw_batch_size 1
!cd {BUNDLE} && python -m monai.bundle run \
    --config_file configs/inference.yaml \
    --bundle_root {BUNDLE} \
    --dataset_dir {TASK_DIR} \
    --data_list_file_path {DL_PATH} \
    --output_dir {PRED_DIR}

In [ ]:
def dice(a, b):
    a, b = a.astype(bool), b.astype(bool)
    s = a.sum() + b.sum()
    return 1.0 if s == 0 else 2.0 * (a & b).sum() / s

rows = []
for f in val_files:
    name = os.path.basename(f['image']).replace('.nii.gz','')
    pred_path = glob.glob(f'{PRED_DIR}/{name}/*.nii.gz')[0]
    P = np.asarray(nib.load(pred_path).dataobj).squeeze()
    G = np.asarray(nib.load(f['label']).dataobj).squeeze()
    assert P.shape == G.shape, (P.shape, G.shape)
    rows.append(dict(case=name, dice_pancreas=round(dice(P>=1, G>=1),3), dice_tumor=round(dice(P==2, G==2),3),
                     gt_tumor_vox=int((G==2).sum()), pred_tumor_vox=int((P==2).sum())))
base_res = pd.DataFrame(rows)
base_res.loc['mean'] = base_res[['dice_pancreas','dice_tumor']].mean().round(3)
base_res.to_csv(f'{RESULTS}/baseline_dints_dice.csv')
base_res

In [ ]:
for f in val_files:
    name = os.path.basename(f['image']).replace('.nii.gz','')
    I = np.asarray(nib.load(f['image']).dataobj); G = np.asarray(nib.load(f['label']).dataobj).squeeze()
    P = np.asarray(nib.load(glob.glob(f'{PRED_DIR}/{name}/*.nii.gz')[0]).dataobj).squeeze()
    area = (G==2).sum(axis=(0,1));  area = area if area.max()>0 else (G>=1).sum(axis=(0,1)); z = int(area.argmax())
    im = np.clip(np.rot90(I[:,:,z]), HU_MIN, HU_MAX)
    fig, ax = plt.subplots(1, 2, figsize=(9, 4.5))
    for a, M, t in zip(ax, [G, P], ['ground truth', 'pretrained DiNTS prediction']):
        m = np.rot90(M[:,:,z]); a.imshow(im, cmap='gray')
        a.contour(m>=1, colors='lime', linewidths=0.9); a.contour(m==2, colors='red', linewidths=0.9)
        a.set_title(t); a.axis('off')
    r = base_res.loc[base_res.case==name].iloc[0]
    fig.suptitle(f'{name} | z={z} | Dice pancreas={r.dice_pancreas} tumor={r.dice_tumor}', fontsize=10)
    plt.tight_layout(); plt.savefig(f'{RESULTS}/baseline_{name}.png', dpi=150); plt.show()

## 6. Deliverables produced by this notebook

In [ ]:
for p in sorted(os.listdir(RESULTS)): print(p)

**Screenshots for the Review 1 deck (item 9 of the circular):**
1. `nvidia-smi` + MONAI config output (environment)
2. Dataset summary printout + `raw_stats_subset.csv` table and `fig_raw_stats.png`
3. Two or three `overlay_*.png` (Stage 6.1 output)
4. `augment_*.png`
5. Bundle run log + `baseline_dints_dice.csv`
6. Two `baseline_*.png` GT-vs-prediction figures (Stage 6.2 baseline)

**Honest statement for the panel:** pipeline validated on MSD Pancreas (20-case subset); the pretrained baseline gives Dice X / Y on our held-out cases; Attention U-Net training and PanTS external validation are the Review 2 targets.

**Next (Review 2):** `02_segmentation_attention_unet.ipynb` — train `monai.networks.nets.AttentionUnet` with `train_tf`, compare to this baseline.